In [18]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [19]:
pdfs = list(Path("./raw/").glob("*.pdf"))

In [20]:
import ollama
import fitz

In [21]:
OLLAMA_URL = "http://localhost:11434"
TEXT_EMBED_MODEL = "nomic-embed-text"
LLM_MODEL = "llama3.2-vision"

In [22]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_ollama import OllamaEmbeddings
from langchain_community.document_loaders import PyPDFLoader

In [23]:
embeddings = OllamaEmbeddings(
    model="qwen3-embedding",
    base_url="http://localhost:11434"
)

In [24]:
text_splitter = SemanticChunker(
    embeddings, 
    breakpoint_threshold_type="percentile" 
)

In [28]:
from tqdm.auto import tqdm

In [25]:
def semantic_chunking():
    embeddings = OllamaEmbeddings(
    model="qwen3-embedding",
    base_url="http://localhost:11434"
    )

    text_splitter = SemanticChunker(
    embeddings, 
    breakpoint_threshold_type="percentile" 
    )
    docs = []
    for doc in pdfs:
        loader = PyPDFLoader(doc)

        docs.extend(loader.load())
        
    semantic_chunks = []
    for doc1 in tqdm(docs, desc="Splitting documents"):
        semantic_chunks.extend(text_splitter.split_documents([doc1]))

    return semantic_chunks

semantic_chunking()







KeyboardInterrupt: 

In [ ]:
import concurrent.futures
from tqdm import tqdm
# Ensure your Langchain/Ollama imports are here too

def load_single_pdf(pdf_path):
    """Helper function to load a single PDF."""
    loader = PyPDFLoader(pdf_path)
    return loader.load()

def chunk_single_doc(doc, text_splitter):
    """Helper function to chunk a single document."""
    return text_splitter.split_documents([doc])

def semantic_chunking(pdfs):
    embeddings = OllamaEmbeddings(
        model="qwen3-embedding",
        base_url="http://localhost:11434"
    )

    text_splitter = SemanticChunker(
        embeddings, 
        breakpoint_threshold_type="percentile" 
    )
    
    docs = []
    # 1. Parallelize PDF Loading
    # ThreadPool is great here because file I/O releases the GIL.
    print("Starting parallel PDF loading...")
    with concurrent.futures.ThreadPoolExecutor() as executor:
        # Map applies the function to the iterable concurrently
        results = list(tqdm(executor.map(load_single_pdf, pdfs), total=len(pdfs), desc="Loading PDFs"))
        for res in results:
            docs.extend(res)
            
    semantic_chunks = []
    
    # 2. Parallelize Semantic Chunking
    # The chunker makes HTTP requests to Ollama, so threads work well.
    # CAUTION: Set max_workers carefully (see note below).
    MAX_WORKERS = 16
    
    print(f"Starting parallel chunking with {MAX_WORKERS} workers...")
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # Submit tasks to the executor
        futures = [executor.submit(chunk_single_doc, doc, text_splitter) for doc in docs]
        
        # as_completed yields futures as they finish, keeping the progress bar accurate
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(docs), desc="Splitting documents"):
            semantic_chunks.extend(future.result())

    return semantic_chunks

semantic_chunks_parallel = semantic_chunking(pdfs)

Starting parallel PDF loading...


Loading PDFs: 100%|██████████| 19/19 [00:30<00:00,  1.60s/it]


Starting parallel chunking with 16 workers...


Splitting documents:  29%|██▊       | 106/371 [3:52:34<9:39:58, 131.32s/it] 

In [5]:
import pickle

# Save the chunks
#with open("semantic_chunks_parallel.pkl", "wb") as f:
#    pickle.dump(semantic_chunks_parallel, f)

# To load them later:
with open("semantic_chunks_parallel.pkl", "rb") as f:
    loaded_chunks = pickle.load(f)


In [17]:
loaded_chunks[100]

Document(metadata={'producer': 'Acrobat Distiller 11.0.9(Windows)', 'creator': '3B2 Total Publishing System 7.51n/W', 'creationdate': '2006-11-06T11:30:28+08:00', 'moddate': '2017-10-11T17:45:28+05:30', 'title': 'nature05180 347..349', 'trapped': '/False', 'source': 'raw/nature05180.pdf', 'total_pages': 4, 'page': 0, 'page_label': '180'}, page_content='Th e  ZGNR s  a r e  a s sum ed  t o  b e  in f in i t e  a l on g  th eyd i r e c t i on . A  sm a l l\nl on g i tud in a l  s ou r c e -d r a in  f i e ld  c ou ld  b e  app l i ed  t o  g en e r a t e  sp in -p o l a r i z ed\ncu r r en t s  a l on g  th eyd i r e c t i on . H yd r o g en  a t om s  on  th e  ed g e s  a r e  d en o t ed  b y\nc i r c l e s . Th e  ZGNR  sh own  in  th i s  f i gu r e  i s  a  1 6 -ZGNR  ( n51 6 ) . Th e  w id thw\no f  ZGNR s  und e r  s tud y  w a s  in  th e  r an g e  1 . 5 – 6 .')